# Optimización de Hiperparámetros con **Optuna**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

In [9]:
# ==============================
# Generar dataset sintético
# ==============================

X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=15,
    n_classes=2,
    random_state=42
)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64)

In [10]:
# ==============================
# Definición dinámica del modelo
# ==============================

class FullyConnectedNN(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout):
        super().__init__()
        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 2))  # salida binaria
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

## Función objetivo para Optuna

La función `objective(trial)` define el proceso que ejecuta en cada *trial* (experimento) para evaluar una combinación de hiperparámetros en una red neuronal **FC**.

### Espacio de búsqueda
Se definen los hiperparámetros que Optuna debe explorar:
- `n_layers`: número de capas ocultas (entre 1 y 3).
- `n_units_l{i}`: número de neuronas por capa (entre 16 y 128).
- `dropout`: tasa de regularización (0.0 a 0.5).
- `lr`: tasa de aprendizaje en escala logarítmica.
- `optimizer`: tipo de optimizador (`Adam` o `SGD`).

### Construcción del modelo
Con los hiperparámetros propuestos por `trial`, se crea una red Fully Connected (`FullyConnectedNN`).  
Se define la función de pérdida `CrossEntropyLoss` y el optimizador correspondiente.

### Entrenamiento
Durante 20 épocas:
- Se realiza *forward propagation*.
- Se calcula la pérdida.
- Se ejecuta *backpropagation* (`loss.backward()`).
- Se actualizan los pesos (`optimizer.step()`).

### Validación
Se evalúa el modelo en el conjunto de validación sin calcular gradientes (`torch.no_grad()`), y se calcula la **accuracy**.

### Salida
La función retorna la `accuracy`, que es la métrica que Optuna intentará **maximizar** para encontrar la mejor combinación de hiperparámetros.

In [11]:
# ==============================
# Función objetivo para Optuna
# ==============================

def objective(trial):

    # ---- Espacio de búsqueda ----
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden_layers = []
    for i in range(n_layers):
        hidden_dim = trial.suggest_int(f"n_units_l{i}", 16, 128)
        hidden_layers.append(hidden_dim)

    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-1)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])

    # ---- Modelo ----
    model = FullyConnectedNN(input_dim=20, hidden_layers=hidden_layers, dropout=dropout)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr)

    # ---- Entrenamiento ----
    epochs = 20
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

    # ---- Validación ----
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            outputs = model(xb)
            _, predicted = torch.max(outputs, 1)
            total += yb.size(0)
            correct += (predicted == yb).sum().item()

    accuracy = correct / total

    return accuracy  # queremos maximizar accuracy

In [12]:
# ==============================
# Ejecutar estudio Optuna
# ==============================

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("\nMejor trial:")
print("  Accuracy:", study.best_trial.value)
print("  Parámetros óptimos:", study.best_trial.params)

[I 2026-02-25 11:16:33,371] A new study created in memory with name: no-name-f9f21781-284c-4f6c-840a-836c8e42b977
C:\Users\Flavio\AppData\Local\Temp\ipykernel_28224\1383826997.py:15: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-1)
[I 2026-02-25 11:16:34,402] Trial 0 finished with value: 0.59 and parameters: {'n_layers': 3, 'n_units_l0': 92, 'n_units_l1': 105, 'n_units_l2': 78, 'dropout': 0.4530891544629699, 'lr': 0.00010449798971736016, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.59.
[I 2026-02-25 11:16:35,249] Trial 1 finished with value: 0.5875 and parameters: {'n_layers': 3, 'n_units_l0': 83, 'n_units_l1': 78, 'n_units_l2': 79, 'dropout': 0.42322537673150346, 'lr': 0.0007419119610918996, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.59.
[I 2026-02-25 11:16:36,220] Tri


Mejor trial:
  Accuracy: 0.9725
  Parámetros óptimos: {'n_layers': 1, 'n_units_l0': 100, 'dropout': 0.26853321694946575, 'lr': 0.002653320382015509, 'optimizer': 'Adam'}


In [13]:
print("\nTodos los trials:")
for trial in study.trials:
    print(trial.number, trial.params, trial.value)


Todos los trials:
0 {'n_layers': 3, 'n_units_l0': 92, 'n_units_l1': 105, 'n_units_l2': 78, 'dropout': 0.4530891544629699, 'lr': 0.00010449798971736016, 'optimizer': 'SGD'} 0.59
1 {'n_layers': 3, 'n_units_l0': 83, 'n_units_l1': 78, 'n_units_l2': 79, 'dropout': 0.42322537673150346, 'lr': 0.0007419119610918996, 'optimizer': 'SGD'} 0.5875
2 {'n_layers': 2, 'n_units_l0': 83, 'n_units_l1': 52, 'dropout': 0.1774182046865615, 'lr': 0.0031083876515100102, 'optimizer': 'Adam'} 0.9675
3 {'n_layers': 3, 'n_units_l0': 64, 'n_units_l1': 67, 'n_units_l2': 72, 'dropout': 0.055309359197079355, 'lr': 0.00012790876050461933, 'optimizer': 'Adam'} 0.8775
4 {'n_layers': 1, 'n_units_l0': 127, 'dropout': 0.09895800091078322, 'lr': 0.07311694910067917, 'optimizer': 'SGD'} 0.935
5 {'n_layers': 1, 'n_units_l0': 68, 'dropout': 0.40218779463938403, 'lr': 0.012315435157727761, 'optimizer': 'SGD'} 0.8425
6 {'n_layers': 1, 'n_units_l0': 67, 'dropout': 0.28184474290298955, 'lr': 0.03500340270456678, 'optimizer': 'Ada

## Con **GridSampler**

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from optuna.samplers import GridSampler
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

# ==============================
# Generar dataset sintético
# ==============================

X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=15,
    n_classes=2,
    random_state=42
)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64)

# ==============================
# Modelo (arquitectura fija)
# ==============================

class FullyConnectedNN(nn.Module):
    def __init__(self, input_dim, dropout):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.model(x)

# ==============================
# Función objetivo para Grid
# ==============================

def objective(trial):

    # Parámetros del grid
    lr = trial.suggest_float("lr", 0.001, 0.01)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    dropout = trial.suggest_float("dropout", 0.1, 0.3)

    model = FullyConnectedNN(input_dim=20, dropout=dropout)

    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr)

    # Entrenamiento
    epochs = 20
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            outputs = model(xb)
            loss = criterion(outputs, yb)
            loss.backward()
            optimizer.step()

    # Validación
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            outputs = model(xb)
            _, predicted = torch.max(outputs, 1)
            total += yb.size(0)
            correct += (predicted == yb).sum().item()

    accuracy = correct / total
    return accuracy

# ==============================
# Definir Grid
# ==============================

search_space = {
    "lr": [0.001, 0.01],
    "optimizer": ["Adam", "SGD"],
    "dropout": [0.1, 0.3]
}

sampler = GridSampler(search_space)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

study.optimize(objective)

# ==============================
# Resultados
# ==============================

print("\nMejor trial:")
print("  Accuracy:", study.best_value)
print("  Parámetros óptimos:", study.best_params)

print("\nTodos los trials:")
for trial in study.trials:
    print(trial.number, trial.params, trial.value)

[I 2026-02-25 11:15:25,958] A new study created in memory with name: no-name-4e8b5868-84f9-41fd-93cd-0afa0157f9dd
[I 2026-02-25 11:15:27,049] Trial 0 finished with value: 0.9525 and parameters: {'lr': 0.01, 'optimizer': 'Adam', 'dropout': 0.3}. Best is trial 0 with value: 0.9525.
[I 2026-02-25 11:15:28,026] Trial 1 finished with value: 0.9725 and parameters: {'lr': 0.01, 'optimizer': 'Adam', 'dropout': 0.1}. Best is trial 1 with value: 0.9725.
[I 2026-02-25 11:15:28,772] Trial 2 finished with value: 0.555 and parameters: {'lr': 0.001, 'optimizer': 'SGD', 'dropout': 0.1}. Best is trial 1 with value: 0.9725.
[I 2026-02-25 11:15:29,500] Trial 3 finished with value: 0.8025 and parameters: {'lr': 0.01, 'optimizer': 'SGD', 'dropout': 0.3}. Best is trial 1 with value: 0.9725.
[I 2026-02-25 11:15:30,238] Trial 4 finished with value: 0.8325 and parameters: {'lr': 0.01, 'optimizer': 'SGD', 'dropout': 0.1}. Best is trial 1 with value: 0.9725.
[I 2026-02-25 11:15:31,174] Trial 5 finished with valu


Mejor trial:
  Accuracy: 0.9725
  Parámetros óptimos: {'lr': 0.01, 'optimizer': 'Adam', 'dropout': 0.1}

Todos los trials:
0 {'lr': 0.01, 'optimizer': 'Adam', 'dropout': 0.3} 0.9525
1 {'lr': 0.01, 'optimizer': 'Adam', 'dropout': 0.1} 0.9725
2 {'lr': 0.001, 'optimizer': 'SGD', 'dropout': 0.1} 0.555
3 {'lr': 0.01, 'optimizer': 'SGD', 'dropout': 0.3} 0.8025
4 {'lr': 0.01, 'optimizer': 'SGD', 'dropout': 0.1} 0.8325
5 {'lr': 0.001, 'optimizer': 'Adam', 'dropout': 0.1} 0.9575
6 {'lr': 0.001, 'optimizer': 'SGD', 'dropout': 0.3} 0.5475
7 {'lr': 0.001, 'optimizer': 'Adam', 'dropout': 0.3} 0.9525
